# ECB Pipeline — demo notebook

This notebook shows how to run the pipeline on your CSV/Excel using `llm.apply_prompt`.

**Free mode by default**: `USE_MOCK=True` in `llm.py` so you can run everything with zero API cost.
When you have credits/billing, set `USE_MOCK=False` and (optionally) `USE_OLLAMA=True` for local models.

In [1]:
import sys, time, importlib
print(sys.version)

t=time.perf_counter(); importlib.import_module("numpy"); print("numpy OK", time.perf_counter()-t)
t=time.perf_counter(); importlib.import_module("pandas"); print("pandas OK", time.perf_counter()-t)


3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
numpy OK 0.06545150000602007
pandas OK 0.25161000015214086


In [2]:
import os, pandas as pd, importlib
import llm
importlib.reload(llm)  # ensure fresh flags
print('USE_MOCK=', llm.USE_MOCK, '| USE_OLLAMA=', llm.USE_OLLAMA)

# --- CONFIG ---

DATA_PATH = r"C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\ecb_speeches_clean_minimal.csv" 
TEXT_COL = "text"


USE_MOCK= False | USE_OLLAMA= True


In [3]:
# Load data (CSV or Excel)
if DATA_PATH.lower().endswith('.xlsx'):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)
print('Loaded shape:', df.shape)

# Ensure the text column exists; if not, pick the first column as a fallback
if TEXT_COL not in df.columns:
    print(f"[warn] '{TEXT_COL}' not in columns. Using first column instead.")
    TEXT_COL = df.columns[0]
df = df.copy()

Loaded shape: (2939, 5)


In [4]:
import sys, importlib, llm_ollama
print("Avant reload ->", llm_ollama.__file__)
llm = importlib.reload(llm_ollama)
print("Après reload ->", llm.__file__)


Avant reload -> c:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\llm_ollama.py
Après reload -> c:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\llm_ollama.py


In [7]:
import requests, time
requests.post("http://127.0.0.1:11434/api/pull", json={"name":"gemma3:4b-instruct"}, timeout=120)
# optional: wait a bit and check
for _ in range(20):
    tags = requests.get("http://127.0.0.1:11434/api/tags", timeout=10).json()
    if any(m.get("model")=="gemma3:4b-instruct" for m in tags.get("models",[])):
        break
    time.sleep(3)


: 

In [5]:
import llm_ollama
llm_ollama.USE_MOCK = False
llm_ollama.USE_OLLAMA = True

out = llm_ollama.apply_prompt(
    df.head(1),
    content_function=lambda t: f'Return JSON: {{"s": "one-sentence summary of: {t[:300]}"}}',
    response_format={...},
    text_column="text",
    sequential=True,
    RPM_BUDGET=0,      # pas de sleep
    MAX_TOKENS=64,     # plus court = plus rapide
    TEMPERATURE=0.0,
    model="gemma3:4b-it-q4_K_M",
)




: 

In [21]:
import math
import pandas as pd
import llm_ollama

# settings
TEXT_COL = "text"
MODEL    = "gemma3:4b-it-q4_K_M"
CHUNK    = 200            # adjust based on your machine
RPM      = 300            # rate limit already enforced in your module
MAXTOK   = 128
TEMP     = 0.0

def summarize_chunk(df_chunk: pd.DataFrame) -> pd.DataFrame:
    out = llm_ollama.apply_prompt(
        df_chunk,
        # we truncate a bit to speed up and control token usage
        content_function=lambda t: f'Return JSON: {{"s": "one-sentence summary of: {t[:800]}"}}',
        response_format={"type":"json_schema","json_schema":{"name":"x","schema":{"type":"object","properties":{"s":{"type":"string"}}},"strict":True}},
        text_column=TEXT_COL,
        sequential=True,         # your implementation is sequential (OK)
        RPM_BUDGET=RPM,
        MAX_TOKENS=MAXTOK,
        TEMPERATURE=TEMP,
        model=MODEL,
        return_dataframe=True,   # return a DataFrame for easier merging
    )
    # normalize the 's' column even if the response wasn't a perfect dict
    if "s" not in out.columns:
        out["s"] = out.apply(lambda r: r.get("s") or r.get("_value") or "", axis=1)
    return out[["s"]]

# --- batch loop ---
results = []
n = len(df)
for start in range(0, n, CHUNK):
    stop = min(start + CHUNK, n)
    part = df.iloc[start:stop]
    print(f"Processing rows {start}:{stop} / {n}")
    out_part = summarize_chunk(part)
    results.append(out_part)

summ = pd.concat(results).reindex(df.index)
df["summary_1line"] = summ["s"].fillna("")

# save outputs
df.to_parquet("ecb_with_summary.parquet", index=False)
df.to_csv("ecb_with_summary.csv", index=False)

print("Done. Sample:")
print(df[["summary_1line"]].head(5))


Processing rows 0:200 / 2939


: 

In [9]:
import pandas as pd
from llm_ollama import apply_prompt

TEXT_COL = "text"
BATCH    = 200

def content_function(t: str) -> str:
    return f'Return JSON: {{"s": "one-sentence summary of: {t[:2000]}"}}'

response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "summary_schema",
        "schema": {"type": "object", "properties": {"s": {"type": "string"}}},
        "strict": True,
    },
}

frames = []
for i in range(0, len(df), BATCH):
    part = df.iloc[i:i+BATCH]
    out_part = apply_prompt(
        part,
        content_function=content_function,
        response_format=response_format,
        text_column=TEXT_COL,
        sequential=True,
        RPM_BUDGET=60,
        MAX_TOKENS=128,
        TEMPERATURE=0.0,
        model="mistral:7b-instruct",
    )
    out_part.to_parquet(f"results_part_{i:06d}.parquet")
    frames.append(out_part)

out_full = pd.concat(frames, axis=0).sort_index()
df_out  = df.copy()
df_out["summary"] = out_full["s"]

df_out.to_csv("results_full_with_summary.csv", index=False)
out_full.to_csv("results_only_summary.csv", index=False)
print("OK -> results_full_with_summary.csv & results_only_summary.csv")
df_out.head()


[warn] row=0 a échoué: RuntimeError — Ollama HTTP 404: {"error":"model 'mistral:7b-instruct' not found"}
[warn] row=1 a échoué: RuntimeError — Ollama HTTP 404: {"error":"model 'mistral:7b-instruct' not found"}
[warn] row=2 a échoué: RuntimeError — Ollama HTTP 404: {"error":"model 'mistral:7b-instruct' not found"}
[warn] row=3 a échoué: RuntimeError — Ollama HTTP 404: {"error":"model 'mistral:7b-instruct' not found"}
[warn] row=4 a échoué: RuntimeError — Ollama HTTP 404: {"error":"model 'mistral:7b-instruct' not found"}
[warn] row=5 a échoué: RuntimeError — Ollama HTTP 404: {"error":"model 'mistral:7b-instruct' not found"}


: 

In [11]:
import importlib, llm
importlib.reload(llm)

def content_function(t: str) -> str:
    # Simple: demande un JSON {"s": "..."} avec un résumé d'une phrase
    return f"Return JSON with one key 's' that summarizes in one sentence: {t[:2000]}"

response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "summary_schema",
        "schema": {"type": "object", "properties": {"s": {"type": "string"}}},
        "strict": True,
    },
}

out_small = llm.apply_prompt(
    df.head(3),
    content_function=content_function,
    response_format=response_format,
    text_column=TEXT_COL,
    sequential=True,
    RPM_BUDGET=300,   # rapide même si MOCK
    MAX_TOKENS=64,
    TEMPERATURE=0,
)

out_small


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# Define your content function and response schema
def content_function(t: str) -> str:
    # Keep it simple for demo: one-sentence summary request
    return f"Return JSON with one key 's' that summarizes in one sentence: {t[:2000]}"

response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "summary_schema",
        "schema": {"type": "object", "properties": {"s": {"type": "string"}}},
        "strict": True,
    },
}

In [ ]:
# Smoke test on 3 rows first (fast & safe)
out_small = llm.apply_prompt(
    df.head(3),
    content_function=content_function,
    response_format=response_format,
    text_column=TEXT_COL,
    sequential=True,
    RPM_BUDGET=300,   # fast
    MAX_TOKENS=64,
    TEMPERATURE=0,
)
out_small

In [ ]:
# Run on a bigger batch (adjust BATCH as you like)
BATCH = 50
frames = []
for i in range(0, len(df), BATCH):
    part = df.iloc[i:i+BATCH]
    res = llm.apply_prompt(
        part,
        content_function=content_function,
        response_format=response_format,
        text_column=TEXT_COL,
        sequential=True,
        RPM_BUDGET=200,
        MAX_TOKENS=96,
        TEMPERATURE=0,
    )
    res.to_parquet(f"results_part_{i:06d}.parquet")
    frames.append(res)

final = pd.concat(frames, axis=0)
final.to_csv("results_full.csv", index=False)
final.head()